# 04 - Metadata Enricher (Weekly)

Auto-enriches tables with:
1. **Auto-generated descriptions** via `ai_query` for uncommented tables
2. **PII detection** via column name pattern matching
3. **Suggested tags** based on schema/table name heuristics

Runs weekly to avoid excessive AI calls.

In [0]:
# Databricks notebook source
import sys as _sys
_nb = (dbutils.notebook.entry_point.getDbutils().notebook()
       .getContext().notebookPath().get())
_sys.path.insert(0, '/Workspace' + '/'.join(_nb.split('/')[:-2]) + '/src')
from lib.common import (
    require_widget, uc_list_tables, uc_list_schemas,
    tables_to_spark, build_exempt_schemas,
    load_exemptions, is_exempt,
)
from lib.policy import load_policy, pii_patterns as _policy_pii
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("control_schema", "uc_hygiene")
dbutils.widgets.text("target_catalogs", "")
dbutils.widgets.text("enable_ai_descriptions", "true")  # set to "false" to skip ai_query calls
dbutils.widgets.text("model_name", "databricks-gemini-3-5-flash")
dbutils.widgets.text("max_ai_descriptions_per_run", "50")


catalog        = require_widget(dbutils, "catalog")
control_schema = require_widget(dbutils, "control_schema")
target_catalogs = [
    c.strip() for c in require_widget(dbutils, "target_catalogs").split(",") if c.strip()
]
enable_ai_descriptions      = dbutils.widgets.get("enable_ai_descriptions").strip().lower() == "true"
model_name                  = dbutils.widgets.get("model_name") or "databricks-gemini-3-5-flash"
max_ai_descriptions_per_run = int(dbutils.widgets.get("max_ai_descriptions_per_run") or "50")
control_fqn                 = f"{catalog}.{control_schema}"

print(f"Control schema:  {control_fqn}")
print(f"Target catalogs: {target_catalogs}")
print(f"AI descriptions: {'enabled' if enable_ai_descriptions else 'DISABLED'}")
print(f"Model:           {model_name if enable_ai_descriptions else 'N/A'}")
print(f"Max AI descriptions per run: {max_ai_descriptions_per_run}")
import time as _t; _task_start = _t.time()


In [0]:
# Function definitions moved to src/lib/common.py — imported in widget cell above.
from databricks.sdk import WorkspaceClient

_sdk = WorkspaceClient()


print("✅ lib.common loaded; SDK client ready.")


In [0]:
import re
from datetime import date
import uuid

scan_id = str(uuid.uuid4())

# PII patterns from governance_policy table (seeded from policies/policy.yml by bootstrap)
_policy = load_policy(spark, catalog, control_schema)
PII_PATTERNS = _policy_pii(_policy)

def detect_pii_type(column_name):
    cn = column_name.lower()
    for pii_type, patterns in PII_PATTERNS.items():
        for pattern in patterns:
            if re.match(pattern, cn):
                return pii_type
    return None

In [0]:
# Step 1: Find tables without comments via UC SDK
_table_rows  = uc_list_tables(_sdk, target_catalogs, control_schema)
_exemptions  = load_exemptions(spark, catalog, control_schema)
_table_rows  = [r for r in _table_rows if not is_exempt(r["catalog_name"], r["schema_name"], r["table_name"], _exemptions)]
_uncommented = [r for r in _table_rows if not r["comment"]]
uncommented  = tables_to_spark(spark, _uncommented)
uncommented_count = len(_uncommented)
print(f"Tables without descriptions across {len(target_catalogs)} catalog(s): {uncommented_count}")


In [0]:
# Step 2: Generate AI descriptions in parallel (ThreadPoolExecutor, not serial loop)
from concurrent.futures import ThreadPoolExecutor, as_completed

tables_to_describe    = _uncommented[:max_ai_descriptions_per_run]
descriptions_generated = 0

if not enable_ai_descriptions:
    print("AI descriptions disabled -- skipping.")
else:
    def _describe_table(row):
        fqn = f"{row['catalog_name']}.{row['schema_name']}.{row['table_name']}"
        try:
            # Get columns via UC SDK — single REST call, no Spark
            ti   = _sdk.tables.get(full_name=fqn)
            cols = [(c.name, c.type_text) for c in (ti.columns or [])[:20]]
            col_desc = ", ".join(f"{c[0]} ({c[1]})" for c in cols) or "no columns found"
            prompt = (
                f"Write a concise 1-2 sentence description for a database table named "
                f"'{row['table_name']}' in schema '{row['schema_name']}' with columns: {col_desc}. "
                f"Just the description, no prefix."
            )
            safe   = " ".join(prompt.replace("'", "''").splitlines())
            result = spark.sql(f"SELECT ai_query('{model_name}', '{safe}') AS description").first().description
            desc   = result.replace("'", "''").strip()[:500]
            spark.sql(f"COMMENT ON TABLE {fqn} IS '{desc}'")
            return fqn, True, None
        except Exception as e:
            return fqn, False, str(e)

    with ThreadPoolExecutor(max_workers=8) as pool:
        futures = [pool.submit(_describe_table, r) for r in tables_to_describe]
        for fut in as_completed(futures):
            fqn, ok, err = fut.result()
            if ok:
                descriptions_generated += 1
            else:
                print(f"  ⚠ {fqn}: {err}")

    print(f"\n✅ Generated descriptions for {descriptions_generated} / {len(tables_to_describe)} tables")


In [0]:
# Step 3: PII detection via column name patterns
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

_col_parts = [
    f"SELECT table_catalog, table_schema, table_name, column_name "
    f"FROM {tc}.information_schema.columns "
    f"WHERE table_schema NOT IN ('information_schema','__databricks_internal','uc_hygiene','uc_hygiene_dev','{control_schema}')"
    for tc in target_catalogs
]
columns_df = spark.sql(" UNION ALL ".join(_col_parts))

@udf(StringType())
def detect_pii_udf(col_name):
    return detect_pii_type(col_name)

pii_candidates = columns_df.withColumn(
    "detected_pii_type", detect_pii_udf(col("column_name"))
).filter(col("detected_pii_type").isNotNull())

pii_count = pii_candidates.count()
print(f"\n🔍 Potential PII columns detected: {pii_count}")
if pii_count > 0:
    pii_candidates.show(20, truncate=False)


In [0]:
# Step 4: Check which PII columns are already tagged (across all target catalogs)
tag_parts = []
for tc in target_catalogs:
    tag_parts.append(f"""
    SELECT catalog_name, schema_name, table_name, column_name
    FROM {tc}.information_schema.column_tags
    WHERE tag_name = 'pii'
    """)

existing_pii_tags = spark.sql("\nUNION ALL\n".join(tag_parts))
existing_pii_tags.createOrReplaceTempView("existing_pii")
pii_candidates.createOrReplaceTempView("pii_candidates")

untagged_pii = spark.sql("""
SELECT pc.*
FROM pii_candidates pc
LEFT JOIN existing_pii ep
  ON pc.table_catalog = ep.catalog_name
  AND pc.table_schema = ep.schema_name
  AND pc.table_name = ep.table_name
  AND pc.column_name = ep.column_name
WHERE ep.column_name IS NULL
""")

untagged_count = untagged_pii.count()
print(f"PII columns needing tags: {untagged_count}")

if untagged_count > 0:
    for row in untagged_pii.limit(100).collect():
        fqn = f"{row.table_catalog}.{row.table_schema}.{row.table_name}"
        try:
            spark.sql(f"SET TAG ON COLUMN {fqn}.{row.column_name} pii = true")
            spark.sql(f"SET TAG ON COLUMN {fqn}.{row.column_name} pii_type = `{row.detected_pii_type}`")
            spark.sql(f"SET TAG ON COLUMN {fqn}.{row.column_name} sensitivity = confidential")
        except Exception as e:
            print(f"  ⚠ Could not tag {fqn}.{row.column_name}: {e}")

    print(f"✅ Auto-tagged up to {min(untagged_count, 100)} PII columns")


In [0]:
# Step 5: Log enrichment actions to control table
spark.sql(f"""
INSERT INTO {catalog}.{control_schema}.scan_results
SELECT
  '{scan_id}', CURRENT_DATE(), 'enrichment', 'table',
  pc.table_catalog, pc.table_schema, pc.table_name, pc.column_name,
  'pii_auto_detected', 'info',
  CONCAT('PII type detected: ', pc.detected_pii_type, ' - auto-tagged'),
  'Verify PII classification is correct',
  NULL, NULL, NULL
FROM pii_candidates pc
""")

print(f"""
========================================
  METADATA ENRICHMENT COMPLETE
========================================
  Scan ID:              {scan_id}
  Descriptions added:   {descriptions_generated}
  PII columns found:    {pii_count}
  PII columns tagged:   {min(untagged_count, 100)}
========================================
""")

In [0]:
# ── Summary & observability ──────────────────────────────────────────────────
total_tables = len(_table_rows)
try:
    pii_count = pii_candidates.count()
except Exception:
    pii_count = 0

print(f"""
{'='*52}
  METADATA ENRICHMENT COMPLETE (WEEKLY)
{'='*52}
  Scan ID:            {scan_id}
  Tables scanned:     {total_tables}
  Without desc:       {uncommented_count}
  AI descriptions:    {descriptions_generated}
  PII candidates:     {pii_count}
  Model:              {model_name if enable_ai_descriptions else 'N/A'}
  Control schema: {catalog}.{control_schema}
{'='*52}
""")

try:
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.job_run_history VALUES (
      CURRENT_DATE(),
      'uc_hygiene_weekly_enrichment',
      'p2_metadata_enricher',
      'p2_enrichment',
      'success',
      {total_tables},
      {pii_count},
      {descriptions_generated},
      int(_t.time() - _task_start),
      'uncommented={uncommented_count} ai_desc={descriptions_generated} pii={pii_count}',
      CURRENT_TIMESTAMP()
    )
    """)
except Exception as _e:
    print(f"Warning: could not write to job_run_history: {_e}")